In [ ]:
import numpy as np
import cv2
import glob

# 체스보드 패턴의 내부 코너 수 (행, 열)
CHECKERBOARD = (7, 10)

# 3D 객체 포인트 및 2D 이미지 포인트를 저장할 배열
objpoints = []  # 실제 세계의 3D 점 (체커보드 기준)
imgpoints = []  # 이미지 평면의 2D 점

# 체커보드 기준 3D 점 (z=0)
objp = np.zeros((CHECKERBOARD[0]*CHECKERBOARD[1], 3), np.float32)
objp[:, :2] = np.mgrid[0:CHECKERBOARD[0], 0:CHECKERBOARD[1]].T.reshape(-1, 2)

# 입력 이미지 경로 (jpg/png 모두 사용 가능)
images = glob.glob('./images/*.jpg')

for fname in images:
    img = cv2.imread(fname)
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    # 코너 찾기
    ret, corners = cv2.findChessboardCorners(gray, CHECKERBOARD, None)

    if ret:
        objpoints.append(objp)
        # 서브픽셀 단위로 코너 정밀화
        criteria = (cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER, 30, 0.001)
        corners2 = cv2.cornerSubPix(gray, corners, (11, 11), (-1, -1), criteria)
        imgpoints.append(corners2)

        # 결과를 이미지로 표시
        img = cv2.drawChessboardCorners(img, CHECKERBOARD, corners2, ret)
        cv2.imshow('Corners', img)
        cv2.waitKey(200)

cv2.destroyAllWindows()

# 실제 캘리브레이션 수행
ret, camera_matrix, dist_coeffs, rvecs, tvecs = cv2.calibrateCamera(
    objpoints, imgpoints, gray.shape[::-1], None, None
)

print("Camera matrix:\n", camera_matrix)
print("\nDistortion coefficients:\n", dist_coeffs)
print("\nRotation vectors:\n", rvecs)
print("\nTranslation vectors:\n", tvecs)

# 결과를 yaml/txt로 저장
import yaml
data = {
    'camera_matrix': camera_matrix.tolist(),
    'dist_coeff': dist_coeffs.tolist(),
}
with open('calibration_result.yaml', 'w') as f:
    yaml.dump(data, f)
